# Tokenization -- Foundations
> Section 01 -- Topic 01 -- foundations

**Prerequisites:** None (first notebook in the curriculum)

**What you'll learn:**
- Why tokenization matters and how it affects model behavior
- How BPE, WordPiece, and Unigram tokenizers work internally
- How SentencePiece unifies these approaches

**What you'll build:**
- A BPE tokenizer from scratch in pure Python

## Setup

We begin by installing and importing the libraries we will use throughout this notebook. The `tokenizers` library from Hugging Face provides fast Rust-backed implementations of all major tokenization algorithms. The `transformers` library gives us access to pretrained tokenizers from popular models like GPT-2, BERT, and Llama. Finally, `sentencepiece` is Google's library for training and using SentencePiece tokenizers, which we will explore at the end of this notebook.

In [ ]:
!pip install -q tokenizers transformers sentencepiece matplotlib numpy

In [ ]:
import re
import math
import collections
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Set

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import AutoTokenizer

## 1. Why Tokenization Matters

Tokenization is the very first step in any language model pipeline. Before a model can process text, it must convert raw strings of characters into a sequence of discrete integer IDs that can be looked up in an embedding table. This seemingly mundane preprocessing step has profound consequences for everything that follows -- from the size of the model's vocabulary and embedding matrix, to its ability to handle multiple languages, to the quality of its outputs on downstream tasks.

Consider what happens when you send a prompt to an LLM. The text you write is first broken into tokens -- subword units that the model has learned to recognize during training. The model never sees raw characters or full words; it sees only these token IDs. If the tokenizer splits a common word into many small pieces, the model must spend capacity learning to reassemble them. If it fails to split rare technical terms, they become "unknown" tokens that carry no useful information.

Tokenization choices affect at least four critical dimensions:
1. **Vocabulary size:** Larger vocabularies mean larger embedding matrices (more parameters to train and store) but fewer tokens per text (faster inference).
2. **Out-of-vocabulary handling:** A poor tokenizer produces many `<UNK>` tokens, destroying information the model needs.
3. **Multilingual performance:** Tokenizers trained mainly on English text may need 3-5x more tokens for the same meaning in other languages, directly increasing cost and reducing quality.
4. **Downstream task quality:** If a tokenizer splits named entities or numbers in unexpected ways, the model struggles to reason about them.

Let's see this concretely by comparing how different models tokenize the same input.

In [ ]:
# Compare tokenization across different models
sample_texts = [
    "The transformer architecture revolutionized NLP.",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "Anticonstitutionnellement is a long French word.",
    "The price is $1,234.56 per unit.",
    "こんにちは世界",  # "Hello world" in Japanese
]

# Load tokenizers from popular models
tokenizer_names = {
    "GPT-2": "gpt2",
    "BERT": "bert-base-uncased",
    "Llama-2": "meta-llama/Llama-2-7b-hf",
}

tokenizers_loaded = {}
for display_name, model_name in tokenizer_names.items():
    try:
        tokenizers_loaded[display_name] = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"Could not load {display_name} ({model_name}): {e}")
        print("  (This may require authentication for gated models like Llama.)")

In [ ]:
# Display how each tokenizer handles the sample texts
for text in sample_texts:
    print(f"\n{'='*80}")
    print(f"Text: {text!r}")
    print(f"{'='*80}")
    for name, tok in tokenizers_loaded.items():
        token_ids = tok.encode(text)
        tokens = tok.convert_ids_to_tokens(token_ids)
        print(f"\n  {name} ({len(token_ids)} tokens):")
        print(f"    Tokens: {tokens}")
        print(f"    IDs:    {token_ids}")

Notice how the same input text produces very different tokenizations. GPT-2 uses byte-level BPE, so it never produces unknown tokens but may split words differently than BERT's WordPiece tokenizer. Japanese text is especially revealing -- a tokenizer trained mostly on English will need many more tokens to represent the same content, directly translating to higher inference cost and potentially worse model performance.

This difference is not just academic. If you are paying per token for an API, tokenizer efficiency directly affects your bill. And if you are training a model, the number of tokens determines how much of your context window a given piece of text consumes.

## 2. Character-Level vs Subword vs Word-Level

Before diving into specific algorithms, let's understand the three broad approaches to tokenization and why the field has converged on subword methods.

**Word-level tokenization** is the most intuitive approach: split text on whitespace and punctuation, and assign each unique word an ID. This works well for English prose but falls apart quickly. English alone has hundreds of thousands of word forms (consider "run", "running", "runner", "ran", etc.), and any word not in the vocabulary becomes an unknown token. For morphologically rich languages like Turkish or Finnish, the vocabulary would need to be enormous to cover even common word forms.

**Character-level tokenization** goes to the other extreme: every character is a token. The vocabulary is tiny (a few hundred entries for Unicode), and there are no unknown tokens. But sequences become very long -- a 1000-word document might become 5000+ character tokens. This means the model must learn to compose characters into meaningful units entirely on its own, which is both slow (quadratic attention cost on longer sequences) and harder to learn.

**Subword tokenization** is the sweet spot. Common words like "the" or "running" are kept as single tokens, while rare words are split into meaningful pieces: "tokenization" might become ["token", "ization"]. The vocabulary is manageable (typically 32K-128K entries), sequences are reasonably short, and there are few or no unknown tokens. This is why virtually every modern LLM uses some form of subword tokenization.

In [ ]:
# Implement naive word-level tokenizer
class NaiveWordTokenizer:
    """A simple word-level tokenizer to demonstrate limitations."""
    
    def __init__(self):
        self.word_to_id: Dict[str, int] = {"<UNK>": 0}
        self.id_to_word: Dict[int, str] = {0: "<UNK>"}
    
    def train(self, corpus: List[str], max_vocab_size: int = 1000):
        """Build vocabulary from corpus, keeping only the most frequent words."""
        word_counts = Counter()
        for text in corpus:
            words = re.findall(r"\b\w+\b|[^\w\s]", text.lower())
            word_counts.update(words)
        
        for word, _ in word_counts.most_common(max_vocab_size - 1):
            idx = len(self.word_to_id)
            self.word_to_id[word] = idx
            self.id_to_word[idx] = word
    
    def encode(self, text: str) -> List[int]:
        words = re.findall(r"\b\w+\b|[^\w\s]", text.lower())
        return [self.word_to_id.get(w, 0) for w in words]
    
    def decode(self, ids: List[int]) -> str:
        return " ".join(self.id_to_word.get(i, "<UNK>") for i in ids)
    
    @property
    def vocab_size(self) -> int:
        return len(self.word_to_id)

In [ ]:
# Implement naive character-level tokenizer
class NaiveCharTokenizer:
    """A simple character-level tokenizer to demonstrate limitations."""
    
    def __init__(self):
        self.char_to_id: Dict[str, int] = {"<UNK>": 0}
        self.id_to_char: Dict[int, str] = {0: "<UNK>"}
    
    def train(self, corpus: List[str]):
        """Build vocabulary from all characters in the corpus."""
        chars = set()
        for text in corpus:
            chars.update(text)
        for ch in sorted(chars):
            idx = len(self.char_to_id)
            self.char_to_id[ch] = idx
            self.id_to_char[idx] = ch
    
    def encode(self, text: str) -> List[int]:
        return [self.char_to_id.get(ch, 0) for ch in text]
    
    def decode(self, ids: List[int]) -> str:
        return "".join(self.id_to_char.get(i, "<UNK>") for i in ids)
    
    @property
    def vocab_size(self) -> int:
        return len(self.char_to_id)

In [ ]:
# Demonstrate the tradeoffs
training_corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing has been revolutionized by transformers.",
    "Tokenization is the first step in any NLP pipeline.",
    "Deep learning models require numerical input representations.",
    "The attention mechanism allows models to focus on relevant parts of the input.",
    "Large language models are trained on massive text corpora.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "Embeddings capture semantic relationships between words.",
]

# Train both tokenizers
word_tok = NaiveWordTokenizer()
word_tok.train(training_corpus, max_vocab_size=100)

char_tok = NaiveCharTokenizer()
char_tok.train(training_corpus)

# Test on seen and unseen text
test_texts = [
    "The quick brown fox.",  # All words in vocabulary
    "Transformerization is not a real word.",  # Made-up word
    "Die schnelle braune Fuchs.",  # German (completely OOV for word tokenizer)
]

for text in test_texts:
    print(f"\nText: {text!r}")
    
    word_ids = word_tok.encode(text)
    char_ids = char_tok.encode(text)
    
    unk_word = sum(1 for i in word_ids if i == 0)
    unk_char = sum(1 for i in char_ids if i == 0)
    
    print(f"  Word-level: {len(word_ids)} tokens, {unk_word} UNK (vocab={word_tok.vocab_size})")
    print(f"    Decoded: {word_tok.decode(word_ids)!r}")
    print(f"  Char-level: {len(char_ids)} tokens, {unk_char} UNK (vocab={char_tok.vocab_size})")
    print(f"    Decoded: {char_tok.decode(char_ids)!r}")

The output above illustrates the fundamental tradeoff. The word-level tokenizer produces short sequences but cannot handle unseen words. The character-level tokenizer handles everything but produces sequences roughly 5x longer. Subword tokenization, which we implement next, gives us the best of both worlds.

## 3. Byte Pair Encoding (BPE)

Byte Pair Encoding is the most widely used subword tokenization algorithm, employed by GPT-2, GPT-3, GPT-4, and many other models. Originally a data compression algorithm from 1994 (Philip Gage), it was adapted for NLP by Sennrich et al. in 2016.

The core idea is simple and elegant:

1. **Start** with a vocabulary of individual characters (or bytes).
2. **Count** all adjacent pairs of tokens in the training corpus.
3. **Merge** the most frequent pair into a single new token.
4. **Repeat** steps 2-3 until the desired vocabulary size is reached.

Each merge creates a new token that represents an increasingly common substring. After training, the sequence of merges defines a deterministic encoding: to tokenize new text, you apply the learned merges in order. Early merges capture things like common letter pairs ("th", "er", "in"), while later merges capture full words ("the", "and") and even multi-word patterns.

Let's implement BPE from scratch to understand exactly how it works.

In [ ]:
class BPETokenizer:
    """A from-scratch implementation of Byte Pair Encoding."""
    
    def __init__(self):
        self.merges: List[Tuple[str, str]] = []  # Ordered list of merge rules
        self.vocab: Dict[str, int] = {}           # Token -> ID mapping
        self.id_to_token: Dict[int, str] = {}     # ID -> Token mapping
    
    def _get_word_freqs(self, corpus: List[str]) -> Dict[Tuple[str, ...], int]:
        """Pre-tokenize corpus into words and count frequencies.
        
        Each word is represented as a tuple of characters with a special
        end-of-word marker '</w>' appended. This marker helps the tokenizer
        distinguish word-final tokens from word-internal ones.
        """
        word_freqs = Counter()
        for text in corpus:
            # Simple whitespace pre-tokenization
            words = text.strip().split()
            for word in words:
                # Convert word to tuple of characters + end marker
                char_tuple = tuple(word) + ("</w>",)
                word_freqs[char_tuple] += 1
        return dict(word_freqs)
    
    def _count_pairs(self, word_freqs: Dict[Tuple[str, ...], int]) -> Counter:
        """Count frequency of all adjacent token pairs across the corpus."""
        pair_counts = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair = (word[i], word[i + 1])
                pair_counts[pair] += freq
        return pair_counts
    
    def _merge_pair(
        self, 
        word_freqs: Dict[Tuple[str, ...], int], 
        pair: Tuple[str, str]
    ) -> Dict[Tuple[str, ...], int]:
        """Apply a merge operation: replace all occurrences of pair with merged token."""
        new_word_freqs = {}
        merged = pair[0] + pair[1]
        
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                # Check if current position matches the pair to merge
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        
        return new_word_freqs
    
    def train(self, corpus: List[str], vocab_size: int = 300, verbose: bool = False):
        """Train BPE tokenizer on a corpus.
        
        Args:
            corpus: List of text strings to train on.
            vocab_size: Target vocabulary size.
            verbose: If True, print each merge operation.
        """
        # Step 1: Get word frequencies with character-level initial tokenization
        word_freqs = self._get_word_freqs(corpus)
        
        # Step 2: Build initial vocabulary from all individual characters
        initial_tokens = set()
        for word in word_freqs:
            for token in word:
                initial_tokens.add(token)
        
        self.vocab = {token: idx for idx, token in enumerate(sorted(initial_tokens))}
        self.merges = []
        
        # Track merge history for visualization
        self.merge_history = []
        
        # Step 3: Iteratively merge most frequent pairs
        num_merges = vocab_size - len(self.vocab)
        for i in range(num_merges):
            pair_counts = self._count_pairs(word_freqs)
            if not pair_counts:
                break
            
            # Find the most frequent pair
            best_pair = pair_counts.most_common(1)[0]
            pair, count = best_pair
            
            if verbose and i < 20:  # Print first 20 merges
                print(f"  Merge {i+1}: {pair[0]!r} + {pair[1]!r} -> {pair[0]+pair[1]!r} (count={count})")
            
            # Record the merge
            self.merges.append(pair)
            self.merge_history.append((pair, count))
            
            # Add merged token to vocabulary
            merged_token = pair[0] + pair[1]
            self.vocab[merged_token] = len(self.vocab)
            
            # Apply merge to all words
            word_freqs = self._merge_pair(word_freqs, pair)
        
        # Build reverse mapping
        self.id_to_token = {v: k for k, v in self.vocab.items()}
        
        if verbose:
            print(f"\n  Final vocabulary size: {len(self.vocab)}")
    
    def encode(self, text: str) -> List[int]:
        """Encode text into token IDs using learned merges."""
        tokens = []
        words = text.strip().split()
        
        for word in words:
            # Start with characters + end-of-word marker
            word_tokens = list(word) + ["</w>"]
            
            # Apply merges in order
            for merge_pair in self.merges:
                i = 0
                new_tokens = []
                while i < len(word_tokens):
                    if (
                        i < len(word_tokens) - 1
                        and word_tokens[i] == merge_pair[0]
                        and word_tokens[i + 1] == merge_pair[1]
                    ):
                        new_tokens.append(merge_pair[0] + merge_pair[1])
                        i += 2
                    else:
                        new_tokens.append(word_tokens[i])
                        i += 1
                word_tokens = new_tokens
            
            tokens.extend(word_tokens)
        
        # Convert tokens to IDs
        return [self.vocab.get(t, 0) for t in tokens]
    
    def encode_to_tokens(self, text: str) -> List[str]:
        """Encode text and return the token strings (for debugging)."""
        tokens = []
        words = text.strip().split()
        
        for word in words:
            word_tokens = list(word) + ["</w>"]
            for merge_pair in self.merges:
                i = 0
                new_tokens = []
                while i < len(word_tokens):
                    if (
                        i < len(word_tokens) - 1
                        and word_tokens[i] == merge_pair[0]
                        and word_tokens[i + 1] == merge_pair[1]
                    ):
                        new_tokens.append(merge_pair[0] + merge_pair[1])
                        i += 2
                    else:
                        new_tokens.append(word_tokens[i])
                        i += 1
                word_tokens = new_tokens
            tokens.extend(word_tokens)
        
        return tokens
    
    def decode(self, ids: List[int]) -> str:
        """Decode token IDs back to text."""
        tokens = [self.id_to_token.get(i, "<UNK>") for i in ids]
        text = "".join(tokens)
        # Replace end-of-word markers with spaces
        text = text.replace("</w>", " ")
        return text.strip()

In [ ]:
# Train our BPE tokenizer
corpus = [
    "the cat sat on the mat",
    "the cat ate the rat",
    "the rat sat on the cat",
    "the dog chased the cat on the mat",
    "a cat and a dog sat on a mat",
    "the cat is the best cat",
    "sitting on the mat is the cat",
    "the mat the cat the rat the dog",
    "natural language processing is interesting",
    "tokenization matters for language models",
    "processing natural language requires tokenization",
    "language models use tokenization for processing text",
    "interesting language processing tokenization methods",
]

bpe = BPETokenizer()
print("Training BPE tokenizer...")
print("First 20 merges:")
bpe.train(corpus, vocab_size=100, verbose=True)

In [ ]:
# Test encoding and decoding
test_texts = [
    "the cat sat on the mat",
    "tokenization is interesting",
    "language processing models",
    "unknown words here",
]

for text in test_texts:
    tokens = bpe.encode_to_tokens(text)
    ids = bpe.encode(text)
    decoded = bpe.decode(ids)
    print(f"\nInput:   {text!r}")
    print(f"Tokens:  {tokens}")
    print(f"IDs:     {ids}")
    print(f"Decoded: {decoded!r}")

In [ ]:
# Visualize: merge history and vocabulary growth
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Merge pair frequencies over time
merge_counts = [count for _, count in bpe.merge_history]
axes[0].bar(range(len(merge_counts)), merge_counts, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Merge operation (ordered)")
axes[0].set_ylabel("Pair frequency")
axes[0].set_title("BPE Merge Pair Frequencies")

# Annotate first few merges
for i, ((p1, p2), count) in enumerate(bpe.merge_history[:5]):
    label = f"{p1!r}+{p2!r}"
    axes[0].annotate(
        label, (i, count), textcoords="offset points",
        xytext=(5, 5), fontsize=7, rotation=45
    )

# Plot 2: Vocabulary size growth
initial_vocab = len(bpe.vocab) - len(bpe.merges)
vocab_sizes = [initial_vocab + i for i in range(len(bpe.merges) + 1)]
axes[1].plot(vocab_sizes, color="darkgreen", linewidth=2)
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Vocabulary size")
axes[1].set_title("Vocabulary Growth During BPE Training")
axes[1].axhline(y=len(bpe.vocab), color="red", linestyle="--", alpha=0.5, label=f"Final: {len(bpe.vocab)}")
axes[1].legend()

plt.tight_layout()
plt.show()

The visualization above reveals an important property of BPE training: early merges have very high frequency (capturing ubiquitous character pairs like "t"+"h" or "e"+"</w>"), while later merges have decreasing frequency. The vocabulary grows linearly -- one new token per merge -- until the target size is reached.

Notice that our from-scratch BPE correctly learns common subwords from the corpus. Words like "the" that appear frequently get merged into single tokens early, while rarer words are split into subword pieces.

## 4. WordPiece

WordPiece, developed at Google for the Japanese/Korean segmentation problem and later used in BERT, looks superficially similar to BPE but differs in a critical way: instead of merging the most *frequent* pair, WordPiece merges the pair that maximizes the *likelihood* of the training data.

Concretely, for a pair (x, y), the merge score is:

$$\text{score}(x, y) = \frac{\text{freq}(xy)}{\text{freq}(x) \times \text{freq}(y)}$$

This is essentially a pointwise mutual information (PMI) measure. A pair of tokens that frequently appear together *relative to their individual frequencies* gets a high score. This means WordPiece prefers to merge tokens that have high mutual information -- they "belong together" -- rather than simply the most common pair.

In practice, WordPiece uses a `##` prefix to indicate that a token is a continuation of a previous token (i.e., it is not the start of a new word). For example, "tokenization" might be encoded as ["token", "##ization"].

In [ ]:
class WordPieceTokenizer:
    """A from-scratch implementation of the WordPiece algorithm."""
    
    def __init__(self):
        self.vocab: Dict[str, int] = {}
        self.id_to_token: Dict[int, str] = {}
    
    def _pre_tokenize(self, corpus: List[str]) -> Dict[Tuple[str, ...], int]:
        """Split corpus into words and represent each as character tuples.
        
        WordPiece uses ## prefix for continuation tokens rather than </w> suffix.
        The first character of each word has no prefix; subsequent characters get ##.
        """
        word_freqs = Counter()
        for text in corpus:
            words = text.strip().split()
            for word in words:
                chars = [word[0]] + [f"##{c}" for c in word[1:]]
                word_freqs[tuple(chars)] += 1
        return dict(word_freqs)
    
    def _compute_pair_scores(
        self, word_freqs: Dict[Tuple[str, ...], int]
    ) -> Dict[Tuple[str, str], float]:
        """Compute WordPiece merge scores based on mutual information."""
        # Count individual token frequencies
        token_freqs = Counter()
        pair_freqs = Counter()
        
        for word, freq in word_freqs.items():
            for i, token in enumerate(word):
                token_freqs[token] += freq
                if i < len(word) - 1:
                    pair_freqs[(word[i], word[i + 1])] += freq
        
        # Compute scores: freq(pair) / (freq(left) * freq(right))
        scores = {}
        for pair, freq in pair_freqs.items():
            scores[pair] = freq / (token_freqs[pair[0]] * token_freqs[pair[1]])
        
        return scores
    
    def _merge_pair(
        self, word_freqs: Dict[Tuple[str, ...], int], pair: Tuple[str, str]
    ) -> Dict[Tuple[str, ...], int]:
        """Merge all occurrences of a pair in the word frequency dict."""
        new_word_freqs = {}
        # The merged token: if second starts with ##, strip the ## prefix
        if pair[1].startswith("##"):
            merged = pair[0] + pair[1][2:]  # Remove ## from second part
        else:
            merged = pair[0] + pair[1]
        
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i + 1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] = freq
        
        return new_word_freqs
    
    def train(self, corpus: List[str], vocab_size: int = 200, verbose: bool = False):
        """Train WordPiece tokenizer."""
        word_freqs = self._pre_tokenize(corpus)
        
        # Build initial vocabulary
        initial_tokens = set()
        for word in word_freqs:
            for token in word:
                initial_tokens.add(token)
        
        self.vocab = {"[UNK]": 0}
        for token in sorted(initial_tokens):
            self.vocab[token] = len(self.vocab)
        
        num_merges = vocab_size - len(self.vocab)
        for i in range(num_merges):
            scores = self._compute_pair_scores(word_freqs)
            if not scores:
                break
            
            best_pair = max(scores, key=scores.get)
            best_score = scores[best_pair]
            
            if verbose and i < 15:
                print(f"  Merge {i+1}: {best_pair[0]!r} + {best_pair[1]!r} (score={best_score:.4f})")
            
            # Create merged token
            if best_pair[1].startswith("##"):
                merged = best_pair[0] + best_pair[1][2:]
            else:
                merged = best_pair[0] + best_pair[1]
            
            self.vocab[merged] = len(self.vocab)
            word_freqs = self._merge_pair(word_freqs, best_pair)
        
        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"\n  Final vocabulary size: {len(self.vocab)}")
    
    def encode(self, text: str) -> List[str]:
        """Encode text using greedy longest-match-first strategy."""
        output_tokens = []
        words = text.strip().split()
        
        for word in words:
            tokens = []
            start = 0
            while start < len(word):
                end = len(word)
                found = False
                while start < end:
                    substr = word[start:end]
                    if start > 0:
                        substr = "##" + substr
                    if substr in self.vocab:
                        tokens.append(substr)
                        found = True
                        break
                    end -= 1
                if not found:
                    tokens.append("[UNK]")
                    start += 1
                else:
                    start = end
            output_tokens.extend(tokens)
        
        return output_tokens

In [ ]:
# Train WordPiece on the same corpus
wp = WordPieceTokenizer()
print("Training WordPiece tokenizer...")
print("First 15 merges:")
wp.train(corpus, vocab_size=100, verbose=True)

In [ ]:
# Compare BPE and WordPiece on the same texts
compare_texts = [
    "the cat sat on the mat",
    "tokenization is interesting",
    "natural language processing",
]

print("BPE vs WordPiece Comparison")
print("=" * 60)
for text in compare_texts:
    bpe_tokens = bpe.encode_to_tokens(text)
    wp_tokens = wp.encode(text)
    print(f"\nText: {text!r}")
    print(f"  BPE       ({len(bpe_tokens):2d} tokens): {bpe_tokens}")
    print(f"  WordPiece ({len(wp_tokens):2d} tokens): {wp_tokens}")

The key difference between BPE and WordPiece is subtle but meaningful. BPE greedily merges the most frequent pair, which tends to produce tokens that reflect surface-level frequency patterns. WordPiece uses a likelihood-based score that considers how much a pair of tokens co-occur relative to their individual frequencies. This can lead to different merge orders and ultimately different vocabularies, even when trained on the same corpus.

In practice, both algorithms produce high-quality subword vocabularies. BPE is used by most GPT-family models, while WordPiece is used by BERT and related models.

## 5. Unigram Language Model

The Unigram model, proposed by Kudo (2018), takes a fundamentally different approach. Instead of starting small and growing the vocabulary through merges, it starts with a *large* initial vocabulary and *prunes* it down to the target size.

The algorithm works as follows:

1. **Initialize** with a large vocabulary of all substrings (up to some length) that appear in the corpus.
2. **Assign probabilities** to each token based on frequency in the corpus.
3. For each token, **compute the loss** that would result from removing it (how much worse would the corpus be tokenized without it?).
4. **Remove** the tokens with the smallest loss impact (keeping a certain percentage).
5. **Repeat** steps 2-4 until the target vocabulary size is reached.

For encoding, the Unigram model finds the tokenization with the highest total probability using the Viterbi algorithm -- a dynamic programming approach that efficiently finds the most likely segmentation.

In [ ]:
class UnigramTokenizer:
    """A from-scratch implementation of the Unigram language model tokenizer."""
    
    def __init__(self):
        self.vocab: Dict[str, float] = {}  # token -> log probability
    
    def _initialize_vocab(
        self, corpus: List[str], max_substring_len: int = 8, initial_vocab_size: int = 500
    ) -> Dict[str, int]:
        """Build initial vocabulary from substring frequencies."""
        substring_freq = Counter()
        for text in corpus:
            words = text.strip().split()
            for word in words:
                word = "_" + word  # Use _ as word-start marker (SentencePiece convention)
                for i in range(len(word)):
                    for j in range(i + 1, min(i + max_substring_len + 1, len(word) + 1)):
                        substring_freq[word[i:j]] += 1
        
        # Keep the most frequent substrings
        # Always keep individual characters
        chars = set()
        for text in corpus:
            chars.update(text)
            chars.add("_")
        
        vocab_freq = {c: substring_freq.get(c, 1) for c in chars}
        for token, freq in substring_freq.most_common(initial_vocab_size):
            vocab_freq[token] = freq
        
        return vocab_freq
    
    def _compute_log_probs(self, vocab_freq: Dict[str, int]) -> Dict[str, float]:
        """Convert frequencies to log probabilities."""
        total = sum(vocab_freq.values())
        return {token: math.log(freq / total) for token, freq in vocab_freq.items()}
    
    def _viterbi_encode(self, word: str, log_probs: Dict[str, float]) -> List[str]:
        """Find the most probable segmentation using Viterbi algorithm."""
        n = len(word)
        # best_score[i] = best log-probability for word[:i]
        best_score = [-float("inf")] * (n + 1)
        best_score[0] = 0.0
        # best_prev[i] = where the last token starts for the best segmentation of word[:i]
        best_prev = [0] * (n + 1)
        
        for i in range(1, n + 1):
            for j in range(max(0, i - 16), i):  # Look back up to 16 chars
                substr = word[j:i]
                if substr in log_probs:
                    score = best_score[j] + log_probs[substr]
                    if score > best_score[i]:
                        best_score[i] = score
                        best_prev[i] = j
        
        # Backtrack to find the best segmentation
        tokens = []
        i = n
        while i > 0:
            j = best_prev[i]
            tokens.append(word[j:i])
            i = j
        
        return list(reversed(tokens))
    
    def _compute_token_loss(
        self, corpus_words: List[str], log_probs: Dict[str, float], token: str
    ) -> float:
        """Estimate the increase in corpus loss if this token were removed."""
        # For efficiency, we compute the loss on a sample
        current_loss = 0.0
        removed_loss = 0.0
        
        temp_log_probs = {k: v for k, v in log_probs.items() if k != token}
        
        for word in corpus_words[:200]:  # Sample for efficiency
            current_tokens = self._viterbi_encode(word, log_probs)
            current_loss += sum(log_probs.get(t, -100) for t in current_tokens)
            
            removed_tokens = self._viterbi_encode(word, temp_log_probs)
            removed_loss += sum(temp_log_probs.get(t, -100) for t in removed_tokens)
        
        return current_loss - removed_loss  # Positive means removing this token hurts
    
    def train(
        self,
        corpus: List[str],
        vocab_size: int = 100,
        shrink_factor: float = 0.8,
        verbose: bool = False
    ):
        """Train Unigram tokenizer by iteratively pruning vocabulary."""
        # Step 1: Initialize large vocabulary
        vocab_freq = self._initialize_vocab(corpus, initial_vocab_size=400)
        
        # Prepare word list with _ prefix
        corpus_words = []
        for text in corpus:
            for word in text.strip().split():
                corpus_words.append("_" + word)
        
        # Identify character-level tokens that must always be kept
        required_tokens = set()
        for word in corpus_words:
            for ch in word:
                required_tokens.add(ch)
        
        iteration = 0
        while len(vocab_freq) > vocab_size:
            iteration += 1
            log_probs = self._compute_log_probs(vocab_freq)
            
            # Compute loss for each token
            token_losses = {}
            for token in vocab_freq:
                if token in required_tokens:
                    token_losses[token] = float("inf")  # Never remove required tokens
                elif len(token) == 1:
                    token_losses[token] = float("inf")
                else:
                    token_losses[token] = self._compute_token_loss(
                        corpus_words, log_probs, token
                    )
            
            # Sort by loss and keep the most important tokens
            target = max(vocab_size, int(len(vocab_freq) * shrink_factor))
            sorted_tokens = sorted(token_losses.items(), key=lambda x: -x[1])
            
            new_vocab_freq = {}
            for token, _ in sorted_tokens[:target]:
                new_vocab_freq[token] = vocab_freq[token]
            
            if verbose:
                print(f"  Iteration {iteration}: {len(vocab_freq)} -> {len(new_vocab_freq)} tokens")
            
            vocab_freq = new_vocab_freq
        
        self.vocab = self._compute_log_probs(vocab_freq)
        if verbose:
            print(f"\n  Final vocabulary size: {len(self.vocab)}")
    
    def encode(self, text: str) -> List[str]:
        """Encode text using Viterbi decoding."""
        tokens = []
        words = text.strip().split()
        for word in words:
            word = "_" + word
            word_tokens = self._viterbi_encode(word, self.vocab)
            tokens.extend(word_tokens)
        return tokens

In [ ]:
# Train Unigram tokenizer
uni = UnigramTokenizer()
print("Training Unigram tokenizer...")
uni.train(corpus, vocab_size=80, verbose=True)

# Test encoding
for text in compare_texts:
    tokens = uni.encode(text)
    print(f"\nText: {text!r}")
    print(f"  Unigram ({len(tokens)} tokens): {tokens}")

The Unigram approach has a distinct advantage: because it uses a probabilistic model, it can naturally handle ambiguous segmentations by choosing the one with the highest total probability. It can also sample different segmentations, which has been used as a data augmentation technique (subword regularization) during training.

The main downside of our from-scratch implementation is that the pruning step is computationally expensive since we need to evaluate the impact of removing each token. In practice, efficient approximations are used.

## 6. SentencePiece

SentencePiece, developed by Kudo and Richardson (2018) at Google, is not a new tokenization algorithm but a framework that unifies BPE and Unigram under a single interface. Its key innovation is treating the input as a raw byte stream rather than pre-tokenized words.

Most tokenizers (including our implementations above) assume the input has already been split into words by whitespace. This is problematic for languages like Chinese, Japanese, and Thai that do not use spaces between words, and for tasks where you want the tokenizer to learn space handling on its own.

SentencePiece solves this by:
1. Treating spaces as regular characters (represented as `_` or the Unicode whitespace character `\u2581`).
2. Operating directly on raw Unicode text with no pre-tokenization.
3. Supporting both BPE and Unigram as the underlying model.

Models like Llama, T5, and ALBERT use SentencePiece. Let's train one using the official library.

In [ ]:
import sentencepiece as spm
import tempfile
import os

# Write corpus to a temporary file (SentencePiece requires file input)
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    for text in corpus:
        f.write(text + "\n")
    # Add more data for better training
    extra_data = [
        "machine learning is a subset of artificial intelligence",
        "deep learning uses neural networks with multiple layers",
        "attention mechanisms transformed the field of NLP",
        "large language models generate human-like text",
        "transformers use self-attention to process sequences",
        "the encoder processes the input while the decoder generates output",
        "pretraining on large corpora followed by fine-tuning on specific tasks",
        "reinforcement learning from human feedback improves model alignment",
    ]
    for text in extra_data:
        f.write(text + "\n")
    corpus_file = f.name

# Train SentencePiece BPE model
model_prefix = tempfile.mktemp()
spm.SentencePieceTrainer.train(
    input=corpus_file,
    model_prefix=model_prefix,
    vocab_size=150,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=3,
)

# Load the trained model
sp = spm.SentencePieceProcessor()
sp.load(model_prefix + ".model")

print(f"SentencePiece vocabulary size: {sp.get_piece_size()}")
print(f"\nSample vocabulary entries:")
for i in range(min(30, sp.get_piece_size())):
    print(f"  {i}: {sp.id_to_piece(i)!r}")

In [ ]:
# Compare SentencePiece output with our from-scratch implementations
test_texts_sp = [
    "the cat sat on the mat",
    "tokenization is interesting",
    "natural language processing",
    "machine learning and deep learning",
]

for text in test_texts_sp:
    sp_tokens = sp.encode_as_pieces(text)
    sp_ids = sp.encode_as_ids(text)
    print(f"\nText: {text!r}")
    print(f"  SentencePiece: {sp_tokens}")
    print(f"  IDs:           {sp_ids}")
    print(f"  Decoded:       {sp.decode(sp_ids)!r}")

# Clean up temp files
os.unlink(corpus_file)
os.unlink(model_prefix + ".model")
os.unlink(model_prefix + ".vocab")

Notice how SentencePiece uses the `\u2581` character (displayed as `_`) to represent spaces. This is a key feature: the tokenizer treats the entire input as a character sequence, and spaces are just another character that can be merged into tokens. This makes it language-agnostic -- it works equally well for English, Japanese, or any other language without requiring language-specific preprocessing.

## 7. Using Hugging Face Tokenizers

In practice, you will rarely implement tokenizers from scratch. The Hugging Face `tokenizers` library provides fast Rust-backed implementations of all the algorithms we have discussed, along with pretrained tokenizers from thousands of models. Understanding the internals (which you now do!) makes you much more effective at using these tools.

The `tokenizers` library organizes the tokenization pipeline into discrete components:
- **Normalizer**: Unicode normalization, lowercasing, etc.
- **Pre-tokenizer**: Initial splitting (whitespace, punctuation, etc.)
- **Model**: The core algorithm (BPE, WordPiece, Unigram)
- **Post-processor**: Adding special tokens (CLS, SEP, BOS, EOS)
- **Decoder**: Converting tokens back to text

Let's explore how real model tokenizers work.

In [ ]:
# Load pretrained tokenizers and compare their behavior
texts_to_compare = [
    "Hello, how are you doing today?",
    "def train(self, corpus, vocab_size=32000):",
    "The U.S. GDP grew by 3.2% in Q4 2024.",
    "I can't believe it's not butter!",
    "supercalifragilisticexpialidocious",
]

for name, tok in tokenizers_loaded.items():
    print(f"\n{'='*60}")
    print(f"{name} tokenizer")
    print(f"  Vocabulary size: {tok.vocab_size}")
    print(f"  Model type: {type(tok).__name__}")
    
    # Show special tokens
    special = {}
    for attr in ["bos_token", "eos_token", "pad_token", "unk_token", "sep_token", "cls_token"]:
        val = getattr(tok, attr, None)
        if val is not None:
            special[attr] = val
    print(f"  Special tokens: {special}")
    print(f"{'='*60}")
    
    for text in texts_to_compare:
        ids = tok.encode(text)
        tokens = tok.convert_ids_to_tokens(ids)
        print(f"\n  Input: {text!r}")
        print(f"  Tokens ({len(tokens)}): {tokens}")

In [ ]:
# Visualize token boundaries for a single text across tokenizers
viz_text = "Tokenization is fundamental to language models."

fig, axes = plt.subplots(len(tokenizers_loaded), 1, figsize=(16, 2.5 * len(tokenizers_loaded)))
if len(tokenizers_loaded) == 1:
    axes = [axes]

colors = plt.cm.Set3(np.linspace(0, 1, 20))

for idx, (name, tok) in enumerate(tokenizers_loaded.items()):
    ax = axes[idx]
    ids = tok.encode(viz_text)
    tokens = tok.convert_ids_to_tokens(ids)
    
    # Draw colored boxes for each token
    x_pos = 0
    for i, token in enumerate(tokens):
        display = token.replace("\u2581", "_").replace("##", "##")
        width = max(len(display) * 0.12, 0.3)
        rect = mpatches.FancyBboxPatch(
            (x_pos, 0), width, 0.6,
            boxstyle="round,pad=0.02",
            facecolor=colors[i % len(colors)],
            edgecolor="gray",
            linewidth=0.5
        )
        ax.add_patch(rect)
        ax.text(
            x_pos + width / 2, 0.3, display,
            ha="center", va="center", fontsize=8, fontfamily="monospace"
        )
        x_pos += width + 0.05
    
    ax.set_xlim(-0.1, x_pos + 0.1)
    ax.set_ylim(-0.1, 0.8)
    ax.set_title(f"{name} ({len(tokens)} tokens)", fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle(f"Token boundaries: {viz_text!r}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Vocabulary overlap analysis between tokenizers
if len(tokenizers_loaded) >= 2:
    tok_names = list(tokenizers_loaded.keys())
    tok_objects = list(tokenizers_loaded.values())
    
    # Get vocabulary sets (just the token strings)
    vocab_sets = {}
    for name, tok in tokenizers_loaded.items():
        vocab = tok.get_vocab()
        vocab_sets[name] = set(vocab.keys())
    
    print("Vocabulary Overlap Analysis")
    print("=" * 50)
    for i, name_a in enumerate(tok_names):
        for name_b in tok_names[i + 1:]:
            set_a = vocab_sets[name_a]
            set_b = vocab_sets[name_b]
            overlap = set_a & set_b
            jaccard = len(overlap) / len(set_a | set_b)
            print(f"\n{name_a} vs {name_b}:")
            print(f"  |{name_a}| = {len(set_a):,}")
            print(f"  |{name_b}| = {len(set_b):,}")
            print(f"  Overlap: {len(overlap):,} tokens")
            print(f"  Jaccard similarity: {jaccard:.3f}")

## Summary

In this notebook, we covered the foundations of tokenization for language models:

1. **Why tokenization matters**: It is the bridge between raw text and the numerical representations that models consume. Tokenization choices directly impact vocabulary size, out-of-vocabulary handling, multilingual performance, and downstream task quality.

2. **Three approaches**: Character-level tokenization has a tiny vocabulary but very long sequences. Word-level tokenization has short sequences but cannot handle unseen words. Subword tokenization is the sweet spot that modern LLMs use.

3. **BPE (Byte Pair Encoding)**: We implemented this from scratch. It starts with characters and iteratively merges the most frequent adjacent pair. Used by GPT-2, GPT-3, GPT-4.

4. **WordPiece**: Similar to BPE but uses a likelihood-based score (mutual information) for merging instead of raw frequency. Uses `##` continuation prefix. Used by BERT.

5. **Unigram**: Takes the opposite approach -- starts large and prunes. Uses Viterbi algorithm for optimal encoding. Used by T5, Llama (via SentencePiece).

6. **SentencePiece**: A framework that wraps BPE and Unigram with raw text processing, eliminating the need for language-specific pre-tokenization.

7. **Hugging Face tokenizers**: Production-ready implementations with Rust-backed speed.

**Next:** [01-tokenization/advanced.ipynb](advanced.ipynb) -- Training custom tokenizers, multilingual challenges, byte-level BPE.